# GSE211713 | Explain observed radiation-associated geometry

**Results-first notebook, from actual public-data ScGeo 0.2.1 outputs.** No H5AD downloads, model fitting, or workstation paths are needed. The complete analysis is the [molecular workflow](https://github.com/liuifrec/scgeo-notebooks/blob/3bc5472defaa6cee8d11edfe49659c072c887117/notebooks/applications/v0.2.0/07_gse211713_shift_explanations.ipynb). This viewer uses a compact, checksum-verified snapshot of its results; the images are rendered from that snapshot and stored in Git.

Source: [GEO GSE211713](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE211713). Five control, four early 17 Gy and four late 17 Gy mice contribute to the three primary cross-sectional contrasts. These are independent groups, not longitudinal measurements from the same mice. Nine state definitions include six major and three refined fibroblast populations; the hierarchies overlap. The original representation includes 131,157 retained cells; the primary contrasts use the qualified cells from 13 mice, not all retained cells.

**Interpretation boundary:** signed molecular accounting on the expression-derived PCA50 space, not causal driver discovery or independent biological validation. Selected genes and plots are retrospective examples. See the [gallery guide](README.md).

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "notebooks/applications/results/gse211713"]
folder = next((p for p in candidates if (p / "selected_results.json").is_file()), None)
if folder is None:
    raise RuntimeError("Open the notebook from its own directory or the repository root.")
raw = (folder / "selected_results.json").read_bytes()
assert hashlib.sha256(raw).hexdigest() == "ce27992692852941726a13ada2b896f8a468aeb824896b308a1e4df45569c331"
data = json.loads(raw)
def table(name, columns):
    frame = pd.DataFrame(data[name], columns=data[columns])
    frame["state"] = frame["state_idx"].map(dict(enumerate(data["states"])))
    if "contrast_idx" in frame:
        frame["contrast"] = frame["contrast_idx"].map(dict(enumerate(data["contrasts"])))
    return frame
summary = table("summary", "summary_columns")
direction = table("directions", "direction_columns")
genes = table("genes", "gene_columns")
genes["gene"] = genes["gene_idx"].map(dict(enumerate(data["gene_names"])))
programs = table("programs", "program_columns")
programs["program"] = programs["program_idx"].map(dict(enumerate(data["program_names"])))
mice = table("mice", "mouse_columns")
assert (len(summary), len(direction), len(genes), len(programs), len(mice)) == (27, 9, 18, 63, 24)
assert not summary.duplicated(["state_idx", "contrast_idx"]).any()
audit = data["mapping_audit"]
assert audit["max_abs_residual"] <= audit["tolerance"]
assert summary.mapping_residual.max() <= audit["state_mapping_policy"]["atol"]
assert (summary.delta_norm - summary.contribution_sum).abs().max() < 2e-5
assert ((genes.expression_delta * genes.axis_loading - genes.contribution).abs() < 3e-9).all()
display(pd.Series({"state-contrast rows": len(summary),
                   "full-data mapping residual (reported)": audit["max_abs_residual"],
                   "full-data acceptance threshold": audit["tolerance"],
                   "maximum absolute rounded sum error": (summary.delta_norm-summary.contribution_sum).abs().max(),
                   "states exceeding original local tolerance": (summary.mapping_residual > summary.default_tolerance).sum()}))

## 1. Geometry and direction

![Displacement magnitudes and directional cosines](figures/01_geometry.svg)

[Open full-size SVG](figures/01_geometry.svg)

The direction cosine is estimated from all 3,500 exported gene-axis loadings, aligned by original feature ID: normalized dot product of the early and late loading axes. Their near-unit lengths are consistent with the recovered nearly orthonormal PCA loadings. This compact snapshot retains the derived cosines and norms, not the full vectors. Recomputing the cosines from the full vectors requires the review archive or a rerun of the molecular workflow.

![Fibroblast displacement direction schematics](figures/02_direction.svg)

[Open full-size SVG](figures/02_direction.svg)

Col13a1-like fibroblasts show a larger early–late directional difference than the other two refined fibroblast populations. The angles are point estimates; no angular uncertainty or temporal cell trajectory is inferred.

In [ ]:
direction["angle_degrees"] = direction.cosine_early_late.map(lambda v: math.degrees(math.acos(v)))
display(summary.pivot(index="state", columns="contrast", values="delta_norm")
        .reindex(data["states"]).round(3))
display(direction[["state", "cosine_early_late", "angle_degrees"]].round(3))

## 2. Signed contribution is not expression change

![Opposite signed Col3a1 contributions with positive expression changes](figures/03_col3a1.svg)

[Open full-size SVG](figures/03_col3a1.svg)

For gene g, its contribution is the robust expression-center contrast multiplied by its loading along the unit displacement direction. The sum over the **full** HVG basis reconstructs the observed magnitude. The selected genes here do not exhaust that sum. Col14a1-like and myofibroblast populations show positive Col3a1 expression-center changes but opposite contribution signs. This is a descriptive geometric distinction, not opposite causal effects.

The curves and shift-embedding plots in the full molecular notebook use the actual ScGeo APIs, including `scgeo.tl.explain_shift` and `scgeo.pl.shift_curves`. This viewer does not invent cell-level curves from aggregate tables.

In [ ]:
early = genes[genes.contrast_idx.eq(0)]
display(early[["state", "gene", "expression_delta", "axis_loading", "contribution",
               "ci_low", "ci_high", "sign_stability"]].round(5))

## 3. Late response and cross-lineage diagnostic

![Selected Spp1 Igkc Sftpc expression and signed contributions](figures/04_late_markers.svg)

[Open full-size SVG](figures/04_late_markers.svg)

Spp1 and Igkc increase while Sftpc decreases; all three support the fitted early-to-late shift in these selected fibroblast states. These markers may reflect multiple cellular sources, ambient RNA, doublets, annotation differences or changing within-state composition. Do not interpret them as proven fibroblast-intrinsic drivers.

The intervals and sign stability are from 200 conditional sample-bootstrap draws on the observed basis and direction; they are not adjusted for retrospective feature selection.

In [ ]:
late = genes[genes.contrast_idx.eq(2)]
net = summary[summary.contrast_idx.eq(2)].set_index("state_idx").delta_norm
selected_share = late.groupby("state_idx").contribution.sum().div(net).mul(100).dropna()
display(late[["state", "gene", "expression_delta", "contribution",
              "ci_low", "ci_high", "sign_stability"]].round(5))
display(selected_share.rename(index=dict(enumerate(data["states"]))).rename("Selected signed sum / net displacement (%)").round(2))

## 4. Program contributions

![Selected predefined program contribution heatmap](figures/05_programs.svg)

[Open full-size SVG](figures/05_programs.svg)

Seven of the 22 usable programs are displayed. The original definitions are restricted to unambiguous measured HVGs; the selected program sizes are retained in the table. These programs can overlap. Their sums are not independent components of a partition, enrichment statistics or causal pathway activity. The epithelial program is a useful diagnostic, not a confirmed intrinsic response.

In [ ]:
display(programs.pivot(index="program", columns=["state", "contrast"], values="contribution").round(3))

## 5. Individual mouse scores

![Unpaired early and late mouse projection scores](figures/06_mouse_scores.svg)

[Open full-size SVG](figures/06_mouse_scores.svg)

Each point is one qualified mouse with a public GEO accession. All four late scores exceed all four early scores in each refined fibroblast population in this fitted-axis analysis. The same mice were used to estimate the direction, so separation is not independent predictive validation. In particular, the Col13a1-like late scores are heterogeneous. The retained cell count differs among mice; dots are not cell-level replicates.

In [ ]:
display(mice[["state", "condition", "public_accession", "n_cells", "shift_score"]].round(4))
for state, frame in mice.groupby("state", sort=False):
    early = frame.loc[frame.condition.eq("17Gy_early"), "shift_score"]
    late = frame.loc[frame.condition.eq("17Gy_late"), "shift_score"]
    assert len(early) == len(late) == 4
    print(state, "| early max:", round(early.max(), 3), "| late min:", round(late.min(), 3))

## 6. What is and is not reproduced here

This results viewer reproduces the tabular comparisons, not the original H5AD-level mapping audit or conditional bootstrap. The figures are deterministic SVGs rendered from the committed compact data by `scripts/render_public_lung_gallery.mjs`; `node scripts/render_public_lung_gallery.mjs --check` from the repository root checks that the snapshots match their source.

For the complete workflow, use the [real-data applications](../../v0.2.0/README.md) and the molecular notebook linked above. Compare cell-level expression and the original cleaned annotations before interpreting cross-lineage markers. A fresh basis excluding markers would be a separate sensitivity analysis, not a change to this frozen result. The archive checksum and extraction procedure are documented in [README](README.md).